# 04-2 — Human-in-the-Loop (HITL) Demo

Demonstrates an agent that **pauses to ask the human for input** at decision points.

The `AskHumanTool` lets the agent:
- Present multiple options with descriptions
- Accept free-form text answers
- Track the full interaction history

In this demo we use `CLIHumanHandler` which reads from `stdin`.  
In production, swap it with the `WebHITLBridge`-backed handler used by the FastAPI server.

**Prerequisites**: `OPENAI_API_KEY` set.

In [ ]:

from ravi.core.agent_catalog import AgentCatalog
from ravi.core.agents.react_agent import ReActAgent
from ravi.integrations.llm.factory import create_model_client
from ravi.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from ravi.core.memory.unbounded_memory import UnboundedMemory
from ravi.core.context.implementations import UnboundedContext
from ravi.catalog.tools.human_input.tool import AskHumanTool, HumanInputHandler, HumanInputRequest, HumanInputResponse, CLIHumanHandler
from ravi.console import Console

from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
API_KEYS = {
    "openai":     settings.OPENAI_API_KEY,
    "anthropic":  settings.ANTHROPIC_API_KEY,
    "google":     settings.GEMINI_API_KEY,
    "groq":       settings.GROQ_API_KEY,
    "openrouter": settings.OPENROUTER_API_KEY,
}

: 

## Run the HITL agent

When the agent calls `ask_human` you will be prompted in the output cell — type your answer and press **Enter**.

In [ ]:
class NotebookMockHandler(HumanInputHandler):
    """Auto-responds with the first available option (for notebook demos)."""

    async def request_input(self, request: HumanInputRequest) -> HumanInputResponse:
        print(f"\n[HITL] Question: {request.question}")
        if request.options:
            chosen = request.options[0]
            print(f"[HITL] Auto-selecting: {chosen.label}")
            return HumanInputResponse(
                selected_option=chosen,
                text=chosen.label,
                is_freeform=False,
            )
        print("[HITL] No options; returning empty freeform.")
        return HumanInputResponse(text="", is_freeform=True)


async def run():
    handler = NotebookMockHandler()
    ask_tool = AskHumanTool(handler=handler, max_requests_per_run=3)

    catalog = AgentCatalog()
    catalog.register_model("primary", create_model_client(CHAT_MODEL, api_keys=API_KEYS))
    catalog.register_memory("default", UnboundedMemory())
    catalog.register_context("default", UnboundedContext())
    for tool in [ask_tool, CalculatorTool(), GetCurrentTimeTool()]:
        catalog.register_tool(tool)

    agent = ReActAgent(
        name="hitl-assistant",
        description="An assistant that asks for human input when needed",
        catalog=catalog,
        system_instructions=(
            "You are a helpful AI assistant. When you need the user's preference or "
            "confirmation, use the ask_human tool to present 2–3 options. "
            "You can ask up to 3 questions per conversation."
        ),
        max_iterations=10,
    )

    print(f"Configured chat model: {CHAT_MODEL}")
    result = await Console(agent).run("Help me plan a team dinner for 8 people this Friday.")

    history = ask_tool.interaction_history
    if history:
        print(f"\n--- Human Interactions ({len(history)}) ---")
        for h in history:
            print(f"  Q: {h['question']}")
            print(f"  A: {h['answer']} (freeform={h['is_freeform']})")

await run()